# 🎬 Piku vs 3 Hyenas — Fast 3D Cartoon Generator (Kaggle)
### ⚡ Dual NVIDIA T4 (30GB VRAM) • Ultra-Fast LTX-Video Pipeline

**⚠️ Kaggle Settings (Right Panel):**
1. **Accelerator:** GPU T4 x 2
2. **Internet:** ON (Required)

## 📦 Step 1: Install & Fetch Storyboard Scenes

In [ ]:
!pip install -q gradio_client ffmpeg-python
!apt-get update -qq && apt-get install -y -qq ffmpeg

!rm -rf videogen
!git clone https://github.com/yashkumarsahu789/videogen.git

import os
assets_dir = "videogen/video_output/piku_vs_hyenas_slapstick/assets"
print("✅ Scene assets ready:", os.listdir(assets_dir))

## 🚀 Step 2: Render All 5 Scenes via Lightricks LTX-Video Cloud Engine

In [ ]:
import os, time, shutil
from pathlib import Path
from gradio_client import Client, handle_file

print("Connecting to LTX-Video Cloud Engine...")
client = Client("Lightricks/ltx-video-distilled")
print("✅ Connected!")

assets_path = Path("videogen/video_output/piku_vs_hyenas_slapstick/assets")
output_dir = Path("output_clips")
output_dir.mkdir(parents=True, exist_ok=True)

scenes = [
    ("scene1_hyenas_leaf_slide.jpg", "clip1.mp4", "Three goofy spotted hyenas comically sliding fast on giant green leaves down dirt path, screaming wide eyed, 3D Pixar cartoon"),
    ("scene2_armadillo_bowling.jpg", "clip2.mp4", "Cheeky baby monkey in coconut helmet laughing mischievously as he bowls a round spiky armadillo down the slope"),
    ("scene3_crocodile_crash.jpg", "clip3.mp4", "Sliding hyenas crash full-speed into the soft round belly of a giant chubby crocodile basking with a smug smirk"),
    ("scene4_giant_spider_launch.jpg", "clip4.mp4", "Giant fuzzy purple cartoon spider emerging from tree trunk, scaring hyenas who launch screaming into mid-air"),
    ("scene5_monkey_victory_laugh.jpg", "clip5.mp4", "Baby monkey Piku giving double thumbs up smiling playfully, three dizzy hyenas hanging upside down from ravine log")
]

clip_paths = []
for img_name, out_name, prompt in scenes:
    img_file = assets_path / img_name
    out_file = output_dir / out_name
    print(f"\n🎬 Rendering {img_name}...")
    t0 = time.time()
    try:
        res = client.predict(
            prompt=prompt,
            negative_prompt="blurry, low quality, jittery, distorted",
            input_image_filepath=handle_file(str(img_file)),
            input_video_filepath=None,
            height_ui=512,
            width_ui=704,
            mode="image-to-video",
            duration_ui=2.5,
            ui_frames_to_use=9,
            seed_ui=42,
            randomize_seed=False,
            ui_guidance_scale=1,
            improve_texture_flag=True,
            api_name="/image_to_video"
        )
        shutil.copyfile(res[0]["video"], str(out_file))
        clip_paths.append(str(out_file))
        print(f"✓ Completed in {round(time.time() - t0, 1)}s -> {out_name}")
    except Exception as e:
        print(f"Error on {img_name}: {e}")

print("\n🎉 ALL SCENES RENDERED!")

## 🎞️ Step 3: Concat & Finalize Master Cartoon Movie

In [ ]:
with open("concat_list.txt", "w") as f:
    for cp in clip_paths:
        f.write(f"file '{os.path.abspath(cp)}'\n")

final_output = "piku_vs_hyenas_3d_master.mp4"
!ffmpeg -y -f concat -safe 0 -i concat_list.txt -c:v libx264 -pix_fmt yuv420p {final_output}
print(f"✅ Final Master Movie Saved: {final_output}")

## 📥 Step 4: Preview & Download

In [ ]:
from IPython.display import HTML, FileLink
from base64 import b64encode

mp4_bytes = open(final_output, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()

display(HTML(f'''
<div style="background:#1E293B; padding:20px; border-radius:16px; border:2px solid #10B981;">
    <h3 style="color:#F59E0B; margin-top:0;">🐵 Piku vs Hyenas 3D Cartoon Master</h3>
    <video width=704 height=512 controls autoplay loop style="border-radius:12px; width:100%;">
        <source src="{data_url}" type="video/mp4">
    </video>
</div>
'''))

print("\n⬇️ Download Final MP4:")
display(FileLink(final_output))